# GPU Dask Cluster Demo

In this notebook, we show how you can do your work on a temporary cluster of GPUs, just as we did with a cluster of CPUs in the "HelioCloud Storage + Burst + SDO" notebook (SDO_Demo.ipynb). The GPU cluster is created using Dask, a tool that allows us to scale up our compute based on the size of the problem we want to solve. 

In [1]:
import dask
from dask.distributed import Client
from dask_gateway import Gateway, GatewayCluster

import os
import pynvml
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf
import numpy as np

In the cell below, we set the options for our cluster. We request the workers on the cluster to use HelioCloud's TensorFlow image. As you may recall from the SDO Demo notebook, a CPU cluster can have up to 100 workers on a HelioCloud instance, allowing you to define how many workers you need for your work.

In the case of a GPU cluster, best practice is to tune the cluster options to the type of instance we are using. For the number of workers, we should use the number of cores on the underlying instance. The default HelioCloud instance only has four cores, so we should set `options.worker_cores = 4`. When we set the worker memory, it is recommended to use about 7/8ths of the underlying instance memory. The HelioCloud instance memory is usually 16 GB, so we should set `options.worker_memory = 14`.

Fortunately, the HelioCloud DaskHub server you are working from already has a default profile with these values set, which we have named "gpu-xlarge." Instead of explicitly setting the number of worker cores and memory, we just include the line `options.profile='gpu-xlarge'`.

In [2]:
gateway = Gateway()
options = gateway.cluster_options()

options.image = 'public.ecr.aws/q3h7b4o8/heliocloud/helio-daskhub-mltf:2025.01.29'
options.profile='gpu-xlarge'

In [ ]:
cluster = gateway.new_cluster(options)
client = cluster.get_client()
n_workers = 3
cluster.scale(n_workers)

cluster

In [ ]:
def get_nvidia_driver_version():
    return pynvml.nvmlSystemGetDriverVersion(), len(tf.config.list_physical_devices('GPU'))

In [ ]:
client.run(get_nvidia_driver_version)

In [ ]:
array_a = np.random.rand(4000,6000).astype(np.float32)
array_b = np.random.rand(6000,4000).astype(np.float32)

In [ ]:
def do_tensor_math(array_a, array_b):
    import os
    # Set log level to 3 to supress INFO and WARNING messages
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
    import tensorflow as tf
    
    num_gpus = len(tf.config.list_physical_devices('GPU'))

    tf.debugging.set_log_device_placement(True)
    
    a = tf.constant(array_a)
    b = tf.constant(array_b)
    #c = tf.matmul(a, b)

    # Run the matrix multiplication 100 times on the CPU
    for i in range(1000):
        c = tf.matmul(a, b)
    
    return c

In [ ]:
%%time
a_scatter = client.scatter(array_a)
b_scatter = client.scatter(array_b)
c = client.submit(do_tensor_math, a_scatter, b_scatter)
#print(c.result())

Start the cluster, figure out how to keep as much of the trickiness outside of the user's sight as possible.

Import tensorflow, initialize our arrays, define our matrix multipication method, make the matrix multiplication loop an outside variable because we want to change it halfway through the notebook.

Start at 1000 loops. Using the standard GPU should take less than a second, using the cluster will take about a second.

Now set it to 10,000 loops. The standard GPU is taking me about 7 minutes. The cluster takes about a second.

change the compute amount, not the memory amount. How many matrix multiplications are we doing?

In [ ]:
c.result()

In [ ]:
cluster.shutdown()

In [ ]:
cluster

### Cell #1

First, do the imports. We're setting the TensorFlow "log level" to 3 so that it supresses warnings, but still outputs whether the TensorFlow operations are taking place on the CPU, or the GPU

In [ ]:
import os
# Set log level to 3 to supress INFO and WARNING messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf
import numpy as np
import time

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

tf.debugging.set_log_device_placement(True)

### Cell #2: Create some tensors

Create the matrices that we'll be working with. TensorFlow requires that the values be in float32 format for doing matrix multiplication on the GPU.

In [ ]:
array_a = np.random.rand(4000,6000).astype(np.float32)
array_b = np.random.rand(6000,4000).astype(np.float32)

### Cell #3: Matrix multiplication on the CPU

We multiply the matrices on the CPU once. In Cell #1, we enabled TensorFlow to log device placement. As a result, we this cell should output the line "Executing op _MklMatMul in device /job:localhost/replica:0/task:0/device:CPU:0" to show that the operation is taking place on the CPU.

In [ ]:
%%time
with tf.device('/CPU:0'):
  # Place tensors on the CPU
  a = tf.constant(array_a)
  b = tf.constant(array_b)
  c = tf.matmul(a, b)

print(c)

### Cell #4: Increase the Processing Demand on the CPU

Now let's do the same matrix multiplication 100 times. We expect that this cell should take around 30 seconds to run. 

In [ ]:
#%%capture
%%time
start_time = time.time()

with tf.device('/CPU:0'):
  # Place tensors on the CPU
  a = tf.constant(array_a)
  b = tf.constant(array_b)
    
  # Run the matrix multiplication 100 times on the CPU
  for i in range(100):
    c = tf.matmul(a, b)

end_time = time.time()
cpu_execution_time = end_time - start_time

### Cell #5: Display CPU Processing Time

Run the cell below to display how many seconds it look for Cell #4 to run. 

In [ ]:
print(f"Execution time on the CPU: {cpu_execution_time} seconds")

### Cell #6: Matrix multiplication on the GPU

Now we do the same calculation on the GPU. The device placement log should show that we are now operating on the GPU.

In [ ]:
%%time
with tf.device('/GPU:0'):
  # Place tensors on the GPU
  a = tf.constant(array_a)
  b = tf.constant(array_b)
  c = tf.matmul(a, b)

print(c)

### Cell #7: Increase the Processing Demand on the GPU

Run the next cell to repeat the same matrix multiplication 100 times. This should take far less time than when we ran it on the CPU.

In [ ]:
%%time
#start_time = time.time()

with tf.device('/GPU:0'):
  # Place tensors on the GPU
  a = tf.constant(array_a)
  b = tf.constant(array_b)
    
  # Run the matrix multiplication 100 times on the GPU
  for i in range(1000):
    c = tf.matmul(a, b)

#print(c)

#end_time = time.time()
#gpu_execution_time = end_time - start_time

In [ ]:
print(c)

### Cell #8: Display GPU Processing Time

Run the cell below to display how many seconds it look for Cell #7 to run. 

In [ ]:
print(f"Execution time on the GPU: {gpu_execution_time} seconds")